# Stage 2 — Augment Dataset v4 (08_augment_dataset_v4)

**Muc dich:** Tao `SeanSunny/items_prompts_tv_4` (~271K train) tu 85K items_prompts_tv_3 bang LLM paraphrase.

**Pipeline:**
1. Merge `items_raw_tv_v6` (full) + `items_tv_v6` (summary) → push `items_tv_v7`
2. Filter price <= 1,000,000 VND → 85,727 items
3. Tinh price bucket → multiplier A5 (5x/3x/2x/1x/4x)
4. LLM Groq Batch rewrite Mo ta + Thong so (giu Tieu de/Danh muc/Thuong hieu)
5. Combine 85K goc + 185K augmented → push `items_prompts_tv_4`

**Khong can GPU. Chi can GROQ_API_KEY va HF_TOKEN.**

In [ ]:
import os
import re
import sys
import json
import time
import pickle
import random
import numpy as np
from pathlib import Path
from collections import Counter, defaultdict
from tqdm.auto import tqdm
from dataclasses import dataclass, field
from typing import Optional

from datasets import load_dataset, DatasetDict, Dataset
from dotenv import load_dotenv
from huggingface_hub import login
from groq import Groq

# --- paths ---
NOTEBOOK_DIR = Path(".")
PRICER_VI_DIR = NOTEBOOK_DIR.parent / "scraping_data_tv" / "Data_processing_for_Vietnamese_data"
sys.path.insert(0, str(PRICER_VI_DIR))
from pricer_vi.items import Item

# --- seed ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- constants ---
SOURCE_RAW    = "SeanSunny/items_raw_tv_v6"
SOURCE_TV6    = "SeanSunny/items_tv_v6"
OUTPUT_TV7    = "SeanSunny/items_tv_v7"
SOURCE_TV3    = "SeanSunny/items_prompts_tv_3"
OUTPUT_TV4    = "SeanSunny/items_prompts_tv_4"

MAX_PRICE     = 1_000_000
QUESTION      = "San pham nay co gia bao nhieu ?"
PRICE_PREFIX  = "\n\nGia la: "

BATCHES_FOLDER = NOTEBOOK_DIR / "batches_aug_v4"
OUTPUT_FOLDER  = NOTEBOOK_DIR / "output_aug_v4"
STATE_FILE     = NOTEBOOK_DIR / "batches_aug_v4.pkl"

BATCHES_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# --- env ---
env_path = NOTEBOOK_DIR.parent / ".env"
load_dotenv(env_path)
HF_TOKEN   = os.environ.get("HF_TOKEN", "")
GROQ_KEY   = os.environ.get("GROQ_API_KEY", "")

if HF_TOKEN:
    login(HF_TOKEN)
    print(f"HF login OK")
else:
    raise RuntimeError("HF_TOKEN not set")

if not GROQ_KEY:
    raise RuntimeError("GROQ_API_KEY not set")

groq_client = Groq(api_key=GROQ_KEY)
print(f"Groq client OK")
print(f"Imports OK")

## 1. Tao items_tv_v7: merge full (raw) + summary (tv6)

In [ ]:
# Load ca 2 datasets
print(f"Loading {SOURCE_RAW}...")
ds_raw = load_dataset(SOURCE_RAW)
print(f"  raw  train={len(ds_raw['train']):,} | val={len(ds_raw['validation']):,} | test={len(ds_raw['test']):,}")

print(f"Loading {SOURCE_TV6}...")
ds_tv6 = load_dataset(SOURCE_TV6)
print(f"  tv6  train={len(ds_tv6['train']):,} | val={len(ds_tv6['validation']):,} | test={len(ds_tv6['test']):,}")

# Verify alignment: title phai khop theo tung vi tri
mismatches = 0
for i in range(min(1000, len(ds_raw['train']))):
    if ds_raw['train'][i]['title'] != ds_tv6['train'][i]['title']:
        mismatches += 1
print(f"Title alignment check (1000 samples): {mismatches} mismatches")
if mismatches > 0:
    print("WARNING: datasets not aligned by index! Pipeline may produce incorrect data.")
else:
    print("Alignment OK: proceeding with index-based merge")

In [ ]:
def merge_split(raw_split, tv6_split) -> list[dict]:
    """Merge full from raw + summary from tv6, keep title/category/price/brand."""
    rows = []
    for raw_row, tv6_row in zip(raw_split, tv6_split):
        rows.append({
            "title":    raw_row["title"],
            "category": raw_row["category"],
            "price":    raw_row["price"],
            "full":     raw_row["full"],       # raw text 2K+ chars
            "brand":    raw_row.get("brand"),
            "summary":  tv6_row["summary"],    # 5-line LLM summary
            "prompt":   None,
            "id":       None,
        })
    return rows

train_v7 = merge_split(ds_raw["train"], ds_tv6["train"])
val_v7   = merge_split(ds_raw["validation"], ds_tv6["validation"])
test_v7  = merge_split(ds_raw["test"], ds_tv6["test"])

print(f"Merged: train={len(train_v7):,} | val={len(val_v7):,} | test={len(test_v7):,}")

# Sanity check: vay sample 0
sample = train_v7[0]
print(f"\nSample 0:")
print(f"  title   : {sample['title'][:80]}")
print(f"  full    : {len(sample['full'] or '')} chars")
print(f"  summary : {sample['summary'][:100] if sample['summary'] else 'NULL'}")
assert sample["full"], "full column is None — alignment issue!"
assert sample["summary"], "summary column is None — alignment issue!"

In [ ]:
# Push items_tv_v7 len HF Hub
# NOTE: dung DatasetDict truc tiep de co split key 'val' (nhat quan voi items_prompts_tv_3)
ds_v7 = DatasetDict({
    "train":      Dataset.from_list(train_v7),
    "validation": Dataset.from_list(val_v7),
    "test":       Dataset.from_list(test_v7),
})

print(f"Pushing {OUTPUT_TV7}...")
ds_v7.push_to_hub(OUTPUT_TV7, private=True)
print(f"Pushed: https://huggingface.co/datasets/{OUTPUT_TV7}")

## 2. Filter + Parse summary

In [ ]:
# Filter train price <= 1M
train_filtered = [row for row in train_v7 if row["price"] <= MAX_PRICE]
print(f"Train after price filter: {len(train_filtered):,} / {len(train_v7):,} items")
print(f"Dropped: {len(train_v7) - len(train_filtered):,} items (price > {MAX_PRICE:,} VND)")

# Gan index trong filtered list (dung cho custom_id batch)
for idx, row in enumerate(train_filtered):
    row["_idx"] = idx

In [ ]:
# Regex parse summary thanh header (3 dong) + body (2 dong)
_FIELD_RE = re.compile(
    r'^(Ti[eê]u \u0111[eê]|Danh m[uụ]c|Th[uươ]ng hi[eệ]u|M[oô] t[aả]|Th[oô]ng s[oố]): ',
    re.MULTILINE | re.UNICODE,
)
_HEADER_KEYS = {'Tiêu đề', 'Danh mục', 'Thương hiệu',
                'Tieu de', 'Danh muc', 'Thuong hieu'}
_BODY_KEYS   = {'Mô tả', 'Thông số', 'Mo ta', 'Thong so'}

def parse_summary(summary: str) -> tuple[str, str] | None:
    """Returns (header, body) or None if parsing fails.
    header = 3 lines: Tieu de / Danh muc / Thuong hieu
    body   = 2 lines: Mo ta / Thong so
    """
    if not summary:
        return None
    header_lines, body_lines = [], []
    for line in summary.strip().split('\n'):
        line = line.strip()
        if not line:
            continue
        if line.startswith('Tiêu đề:') or line.startswith('Tieu de:'):
            header_lines.append(line)
        elif line.startswith('Danh mục:') or line.startswith('Danh muc:'):
            header_lines.append(line)
        elif line.startswith('Thương hiệu:') or line.startswith('Thuong hieu:'):
            header_lines.append(line)
        elif line.startswith('Mô tả:') or line.startswith('Mo ta:'):
            body_lines.append(line)
        elif line.startswith('Thông số:') or line.startswith('Thong so:'):
            body_lines.append(line)
    if len(header_lines) == 3 and len(body_lines) == 2:
        return '\n'.join(header_lines), '\n'.join(body_lines)
    return None

# Verify tren 100 samples
parse_ok = parse_fail = 0
for row in train_filtered[:1000]:
    result = parse_summary(row["summary"])
    if result:
        parse_ok += 1
    else:
        parse_fail += 1

print(f"Parse check (1000 samples): OK={parse_ok} FAIL={parse_fail}")
if parse_fail > 0:
    print("WARNING: some summaries have unexpected format!")
    for row in train_filtered[:1000]:
        if parse_summary(row["summary"]) is None:
            print(f"  FAIL summary: {repr(row['summary'][:200])}")
            break
else:
    print("Parse OK")
    header0, body0 = parse_summary(train_filtered[0]["summary"])
    print(f"\nSample header:\n{header0}")
    print(f"\nSample body:\n{body0}")

## 3. Price bucket distribution + multiplier A5

In [ ]:
# Phan tich distribution va tinh tong requests theo multiplier
BUCKETS = [
    ("<50K",     0,          50_000,    5),
    ("50-100K",  50_000,    100_000,    3),
    ("100-200K", 100_000,   200_000,    2),
    ("200-500K", 200_000,   500_000,    1),
    ("500K-1M",  500_000, 1_000_001,    4),
]

prices = np.array([row["price"] for row in train_filtered])
total_items = len(train_filtered)

print(f"{'Bucket':<12} {'Items':>7} {'%':>6} {'Multiplier':>11} {'Augmented':>10} {'Total (orig+aug)':>17}")
print("-" * 70)

total_augmented = 0
bucket_multipliers = {}   # idx -> n_versions

for name, lo, hi, mult in BUCKETS:
    mask = (prices >= lo) & (prices < hi)
    count = int(mask.sum())
    aug = count * mult
    total_augmented += aug
    pct = count / total_items * 100
    print(f"{name:<12} {count:>7,} {pct:>5.1f}%   {mult:>5}x        {aug:>9,}     {count + aug:>10,}")

    # Gan multiplier cho tung item
    indices_in_bucket = np.where(mask)[0]
    for idx in indices_in_bucket:
        bucket_multipliers[int(idx)] = mult

total_train_tv4 = total_items + total_augmented
print("-" * 70)
print(f"{'TOTAL':<12} {total_items:>7,}        {'avg=' + f'{total_augmented/total_items:.1f}x':>11} {total_augmented:>9,}     {total_train_tv4:>10,}")
print(f"\nTarget range: 255K-300K. Actual: {total_train_tv4:,}")

# Verify tat ca items da co multiplier
assert len(bucket_multipliers) == total_items, "Some items missing multiplier!"
print(f"Multiplier assigned: {len(bucket_multipliers):,} items OK")

## 4. AugBatch class + SYSTEM_PROMPT

In [ ]:
MODEL = "openai/gpt-oss-20b"
BATCH_SIZE = 1_000

SYSTEM_PROMPT_AUG = """Dua vao thong tin san pham goc va tom tat hien tai, hay viet lai 'Mo ta' va 'Thong so' theo cach khac.
Yeu cau:
- Giu nguyen y nghia, nhung dung tu ngu va cach dien dat khac.
- Khong duoc viet giong voi Mo ta/Thong so hien tai.
- Chi tra loi dung 2 dong theo dinh dang sau, khong them gi khac:
Mo ta: [1 cau mo ta san pham]
Thong so: [1 cau ve tinh nang noi bat]"""


def build_user_message(full_text: str, body: str) -> str:
    """LLM input: raw product text + existing Mo ta + Thong so."""
    return f"Thong tin san pham goc:\n{full_text}\n\n---\nTom tat hien tai:\n{body}"


@dataclass
class AugBatch:
    """1 Groq Batch file (1000 requests max)."""
    start: int
    end: int
    filename: str
    file_id: Optional[str] = None
    batch_id: Optional[str] = None
    output_file_id: Optional[str] = None
    done: bool = False

    # request_list: list of (item_idx, version, user_msg) — NOT pickled (too large)


class AugBatchManager:
    """Manages all AugBatch objects for the full augmentation job."""

    batches: list[AugBatch] = []
    # request_list: flat list of (item_idx, version, user_msg)
    request_list: list[tuple] = []

    @classmethod
    def build_requests(cls, filtered_items: list[dict], multipliers: dict) -> None:
        """Build flat request list from filtered items and multipliers."""
        cls.request_list = []
        skipped = 0
        for idx, row in enumerate(tqdm(filtered_items, desc="Building requests")):
            n_versions = multipliers[idx]
            result = parse_summary(row["summary"])
            if result is None:
                skipped += 1
                continue
            _, body = result
            full_text = row["full"] or ""
            for v in range(n_versions):
                user_msg = build_user_message(full_text, body)
                cls.request_list.append((idx, v, user_msg))
        print(f"Total requests: {len(cls.request_list):,} | Skipped (bad summary): {skipped}")

    @classmethod
    def create_batches(cls) -> None:
        """Slice request_list into AugBatch objects."""
        cls.batches = []
        total = len(cls.request_list)
        for start in range(0, total, BATCH_SIZE):
            end = min(start + BATCH_SIZE, total)
            filename = f"aug_{start}_{end}.jsonl"
            cls.batches.append(AugBatch(start=start, end=end, filename=filename))
        print(f"Created {len(cls.batches)} batches ({BATCH_SIZE} requests each)")

    @classmethod
    def _make_jsonl_line(cls, item_idx: int, version: int, user_msg: str) -> str:
        custom_id = f"{item_idx}_{version}"
        body = {
            "model": MODEL,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT_AUG},
                {"role": "user",   "content": user_msg},
            ],
            "reasoning_effort": "low",
        }
        return json.dumps({
            "custom_id": custom_id,
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": body,
        }, ensure_ascii=False)

    @classmethod
    def write_and_submit(cls, batch: AugBatch) -> None:
        """Write JSONL + upload file + submit batch job."""
        fpath = BATCHES_FOLDER / batch.filename
        with fpath.open("w", encoding="utf-8") as f:
            for item_idx, v, user_msg in cls.request_list[batch.start:batch.end]:
                f.write(cls._make_jsonl_line(item_idx, v, user_msg))
                f.write("\n")
        with fpath.open("rb") as f:
            resp = groq_client.files.create(file=f, purpose="batch")
        batch.file_id = resp.id
        resp2 = groq_client.batches.create(
            completion_window="24h",
            endpoint="/v1/chat/completions",
            input_file_id=batch.file_id,
        )
        batch.batch_id = resp2.id

    @classmethod
    def run(cls) -> None:
        """Submit all batches."""
        for batch in tqdm(cls.batches, desc="Submitting batches"):
            if not batch.batch_id:
                cls.write_and_submit(batch)
        print(f"Submitted {len(cls.batches)} batches")

    @classmethod
    def fetch(cls) -> int:
        """Check status and download completed batches. Returns finished count."""
        for batch in cls.batches:
            if batch.done:
                continue
            result = groq_client.batches.retrieve(batch.batch_id)
            if result.status == "completed":
                batch.output_file_id = result.output_file_id
                out_path = str(OUTPUT_FOLDER / batch.filename)
                groq_client.files.content(batch.output_file_id).write_to_file(out_path)
                batch.done = True
        finished = sum(1 for b in cls.batches if b.done)
        print(f"Finished {finished} / {len(cls.batches)} batches")
        return finished

    @classmethod
    def save(cls) -> None:
        """Pickle batch state (without request_list — too large)."""
        with STATE_FILE.open("wb") as f:
            pickle.dump(cls.batches, f)
        print(f"State saved: {STATE_FILE} ({len(cls.batches)} batches)")

    @classmethod
    def load(cls) -> None:
        """Load batch state from pickle."""
        with STATE_FILE.open("rb") as f:
            cls.batches = pickle.load(f)
        print(f"State loaded: {len(cls.batches)} batches")

    @classmethod
    def resubmit_failed(cls) -> int:
        """Resubmit batches that failed/expired/cancelled."""
        resubmitted = 0
        for batch in cls.batches:
            if batch.done:
                continue
            result = groq_client.batches.retrieve(batch.batch_id)
            if result.status in ("failed", "expired", "cancelled"):
                cls.write_and_submit(batch)
                resubmitted += 1
                time.sleep(0.5)
        if resubmitted > 0:
            cls.save()
        print(f"Resubmitted {resubmitted} batches")
        return resubmitted


print("AugBatchManager defined OK")
print(f"MODEL: {MODEL}")
print(f"SYSTEM_PROMPT_AUG (preview): {SYSTEM_PROMPT_AUG[:100]}...")

## 5. Test don le — 1 item voi Groq single API

In [ ]:
# Test 3 item don le tu cac bucket khac nhau truoc khi submit batch
test_idxs = [0, 1000, 50000]  # khoang <50K, 100-200K, 200-500K

for tidx in test_idxs:
    row = train_filtered[tidx]
    result = parse_summary(row["summary"])
    if result is None:
        print(f"[{tidx}] SKIP: bad summary")
        continue
    header, body = result
    user_msg = build_user_message(row["full"] or "", body)

    resp = groq_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_AUG},
            {"role": "user",   "content": user_msg},
        ],
        reasoning_effort="low",
    )
    llm_out = resp.choices[0].message.content

    print(f"\n[{tidx}] Category: {row['category']} | Price: {row['price']:,} VND")
    print(f"  Title  : {row['title'][:80]}")
    print(f"  ORIGINAL body:\n    {body.replace(chr(10), chr(10)+'    ')}")
    print(f"  LLM output:\n    {llm_out.strip().replace(chr(10), chr(10)+'    ')}")
    print(f"  Tokens : {resp.usage.prompt_tokens} in / {resp.usage.completion_tokens} out")

## 6. Test batch — 15 samples (Groq Batch API)

**[USER]** Kiem tra ket qua test batch truoc khi chay full.
Neu chat luong paraphrase tot → chay cell 'Full batch'.
Neu SYSTEM_PROMPT can chinh → sua cell-08-12 roi chay lai tu day.

In [ ]:
TEST_N = 15  # so sample test
TEST_IDXS = list(range(0, len(train_filtered), len(train_filtered) // TEST_N))[:TEST_N]

# Build test JSONL
test_jsonl_path = BATCHES_FOLDER / "test_batch.jsonl"
test_requests = []  # (item_idx, version=0, user_msg)

for idx in TEST_IDXS:
    row = train_filtered[idx]
    result = parse_summary(row["summary"])
    if result is None:
        continue
    _, body = result
    user_msg = build_user_message(row["full"] or "", body)
    test_requests.append((idx, 0, user_msg))

with test_jsonl_path.open("w", encoding="utf-8") as f:
    for item_idx, v, user_msg in test_requests:
        f.write(AugBatchManager._make_jsonl_line(item_idx, v, user_msg))
        f.write("\n")

# Upload + submit
with test_jsonl_path.open("rb") as f:
    test_file = groq_client.files.create(file=f, purpose="batch")
test_batch_job = groq_client.batches.create(
    completion_window="24h",
    endpoint="/v1/chat/completions",
    input_file_id=test_file.id,
)
print(f"Test batch submitted: {test_batch_job.id}")
print(f"Requests: {len(test_requests)}")

In [ ]:
# Poll test batch
test_output_path = OUTPUT_FOLDER / "test_batch.jsonl"
while True:
    result = groq_client.batches.retrieve(test_batch_job.id)
    if result.status == "completed":
        groq_client.files.content(result.output_file_id).write_to_file(str(test_output_path))
        print("Test batch DONE")
        break
    elif result.status in ("failed", "expired", "cancelled"):
        raise RuntimeError(f"Test batch {result.status}")
    print(f"Status: {result.status} — waiting 15s...")
    time.sleep(15)

In [ ]:
def parse_aug_output(llm_text: str) -> tuple[str, str] | None:
    """Parse LLM 2-line output. Returns (mo_ta_text, thong_so_text) or None."""
    mo_ta = thong_so = None
    for line in llm_text.strip().split('\n'):
        line = line.strip()
        if line.startswith('Mô tả:') or line.startswith('Mo ta:'):
            mo_ta = re.sub(r'^(M[oô] t[aả]):?\s*', '', line).strip()
        elif line.startswith('Thông số:') or line.startswith('Thong so:'):
            thong_so = re.sub(r'^(Th[oô]ng s[oố]):?\s*', '', line).strip()
    if mo_ta and thong_so:
        return mo_ta, thong_so
    return None


# Hien thi ket qua test batch
print(f"=== Test Batch Results ({TEST_N} samples) ===")
parse_fail_count = 0

with test_output_path.open(encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        cid = obj["custom_id"]
        item_idx = int(cid.split("_")[0])
        llm_text = obj["response"]["body"]["choices"][0]["message"]["content"]

        row = train_filtered[item_idx]
        result = parse_summary(row["summary"])
        header, orig_body = result

        parsed = parse_aug_output(llm_text)
        if parsed is None:
            parse_fail_count += 1
            print(f"\n[{item_idx}] PARSE FAIL: {repr(llm_text[:100])}")
            continue

        mo_ta, thong_so = parsed
        print(f"\n[{item_idx}] {row['title'][:60]} | {row['price']:,} VND")
        print(f"  ORIG  Mo ta   : {orig_body.split(chr(10))[0]}")
        print(f"  ORIG  Thong so: {orig_body.split(chr(10))[1] if chr(10) in orig_body else ''}")
        print(f"  NEW   Mo ta   : Mô tả: {mo_ta}")
        print(f"  NEW   Thong so: Thông số: {thong_so}")

print(f"\nParse failures: {parse_fail_count}/{TEST_N}")
print("\n[USER] Kiem tra chat luong paraphrase o tren.")
print("       Neu tot → chay Full batch.")
print("       Neu can chinh → sua SYSTEM_PROMPT_AUG roi chay lai tu cell-08-12.")

## 7. Full batch — 185K requests (Groq Batch API)

**[USER] Chi chay sau khi confirm test batch OK.**

In [ ]:
# Build request list
AugBatchManager.build_requests(train_filtered, bucket_multipliers)
AugBatchManager.create_batches()
print(f"\nEst. cost: input ~{len(AugBatchManager.request_list)*750/1e6:.1f}M tokens")
print(f"           @ ~$0.06/1M input + $0.60/1M output via Groq Batch")

In [ ]:
# Submit tat ca batches
AugBatchManager.run()

In [ ]:
# QUAN TRONG: Save state ngay sau submit — neu kernel crash mat batch_ids
AugBatchManager.save()

In [ ]:
# Poll cho den khi xong (~ 1-6 gio)
while True:
    finished = AugBatchManager.fetch()
    if finished == len(AugBatchManager.batches):
        print("All batches DONE!")
        break
    print(f"Waiting 60s... ({finished}/{len(AugBatchManager.batches)})")
    time.sleep(60)

In [ ]:
AugBatchManager.save()  # save final state

### Resume (neu kernel crash)
```python
# AugBatchManager.load()
# AugBatchManager.build_requests(train_filtered, bucket_multipliers)  # re-build request_list
# AugBatchManager.fetch()
```

In [ ]:
# Resubmit failed batches (neu co)
AugBatchManager.resubmit_failed()

## 8. Parse ket qua + Build augmented prompts

In [ ]:
# Thu thap tat ca LLM outputs tu output folder
aug_results = {}   # (item_idx, version) -> llm_text
parse_errors = 0

for batch in tqdm(AugBatchManager.batches, desc="Reading outputs"):
    out_path = OUTPUT_FOLDER / batch.filename
    if not out_path.exists():
        print(f"WARNING: missing output file {batch.filename}")
        continue
    with out_path.open(encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            cid = obj["custom_id"]
            parts = cid.split("_")
            item_idx, version = int(parts[0]), int(parts[1])
            llm_text = obj["response"]["body"]["choices"][0]["message"]["content"]
            aug_results[(item_idx, version)] = llm_text

print(f"Total LLM outputs collected: {len(aug_results):,}")
print(f"Expected: {len(AugBatchManager.request_list):,}")
missing = len(AugBatchManager.request_list) - len(aug_results)
print(f"Missing: {missing}")

In [ ]:
# Build augmented examples
QUESTION_FULL = "Sản phẩm này có giá bao nhiêu ?"
PRICE_PREF    = "\n\nGiá là: "

augmented_examples = []
build_ok = build_fail = 0

for (item_idx, version), llm_text in tqdm(aug_results.items(), desc="Building prompts"):
    row = train_filtered[item_idx]
    result = parse_summary(row["summary"])
    if result is None:
        build_fail += 1
        continue
    header, _ = result

    parsed = parse_aug_output(llm_text)
    if parsed is None:
        build_fail += 1
        continue

    mo_ta, thong_so = parsed
    new_body = f"Mô tả: {mo_ta}\nThông số: {thong_so}"
    new_summary = f"{header}\n{new_body}"
    new_prompt  = f"{QUESTION_FULL}\n{new_summary}{PRICE_PREF}"

    augmented_examples.append({
        "prompt":         new_prompt,
        "completion":     str(int(round(row["price"] / 1000))),
        "price_vnd_true": int(row["price"]),
    })
    build_ok += 1

print(f"Augmented examples built: {build_ok:,} OK | {build_fail} failed")

In [ ]:
# Sample check: xem 5 augmented examples
sample_indices = random.sample(range(len(augmented_examples)), min(5, len(augmented_examples)))
for si in sample_indices:
    ex = augmented_examples[si]
    print(f"[{si}] completion={ex['completion']} | price={ex['price_vnd_true']:,} VND")
    print(ex['prompt'][:300])
    print("---")

## 9. Combine original + augmented + Push items_prompts_tv_4

In [ ]:
# Load original items_prompts_tv_3 (3 splits)
print(f"Loading {SOURCE_TV3}...")
ds_tv3 = load_dataset(SOURCE_TV3)
orig_train = list(ds_tv3["train"])
orig_val   = list(ds_tv3["val"])
orig_test  = list(ds_tv3["test"])
print(f"  original train={len(orig_train):,} | val={len(orig_val):,} | test={len(orig_test):,}")

# Verify schema
assert set(orig_train[0].keys()) == {"prompt", "completion", "price_vnd_true"}, \
    f"Unexpected schema: {orig_train[0].keys()}"
print("Schema OK: prompt, completion, price_vnd_true")

In [ ]:
# Combine + shuffle train
combined_train = orig_train + augmented_examples
random.seed(SEED)
random.shuffle(combined_train)

print(f"Combined train: {len(orig_train):,} (orig) + {len(augmented_examples):,} (aug) = {len(combined_train):,}")
print(f"Val  (unchanged): {len(orig_val):,}")
print(f"Test (unchanged): {len(orig_test):,}")

# Sample manual check: 50 items
manual_sample = random.sample(combined_train, 50)
print(f"\nManual sample 50 — price range check:")
prices_sample = [ex["price_vnd_true"] for ex in manual_sample]
print(f"  min={min(prices_sample):,} | max={max(prices_sample):,} | mean={sum(prices_sample)/len(prices_sample):,.0f}")

# Check no prompt is empty
empty_prompts = sum(1 for ex in combined_train if not ex["prompt"])
empty_completions = sum(1 for ex in combined_train if not ex["completion"])
print(f"\nEmpty prompts: {empty_prompts} | Empty completions: {empty_completions}")
assert empty_prompts == 0 and empty_completions == 0, "Found empty prompts/completions!"
print("Data quality OK")

In [ ]:
# Push items_prompts_tv_4
ds_tv4 = DatasetDict({
    "train": Dataset.from_list(combined_train),
    "val":   Dataset.from_list(orig_val),
    "test":  Dataset.from_list(orig_test),
})

print(f"Pushing {OUTPUT_TV4}...")
print(ds_tv4)
ds_tv4.push_to_hub(OUTPUT_TV4, private=True)
print(f"Pushed: https://huggingface.co/datasets/{OUTPUT_TV4}")
print(f"\nDone! Train size: {len(combined_train):,} (~{len(combined_train)/1000:.0f}K)")

## Leaderboard dataset

| Dataset | Train | Val | Test | Note |
|---|---|---|---|---|
| items_prompts_tv_3 | 85,727 | 3,926 | 3,872 | Goc |
| **items_prompts_tv_4** | **~271K** | **3,926** | **3,872** | Aug x5/3/2/1/4 theo bucket |

**Buoc tiep:** Chay `06_train_v4_scratch.ipynb` tren RTX 5090 32GB voi items_prompts_tv_4.